# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", getattr(metadata, 'name', None))
print("Description:", getattr(metadata, 'description', None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all record sets (using their `@id`), then for each, their columns/fields.

In [ ]:
# List all record sets available in the dataset by their @id
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    if isinstance(metadata.recordSet, list):
        record_sets = metadata.recordSet
    else:
        record_sets = [metadata.recordSet]
else:
    print("No record sets found in metadata. Attempting to load automatically detected ones...")
    # Attempt loading record set IDs from the dataset
    record_sets = dataset._list_record_sets()  # Fallback internal API (not guaranteed)

print(f'Found record sets:')
all_record_set_ids = []
for recset in record_sets:
    # recset could be a string @id or an object with @id
    if isinstance(recset, dict):
        rec_id = recset.get('@id', str(recset))
    else:
        rec_id = str(recset)
    print(f'  RecordSet @id: {rec_id}')
    all_record_set_ids.append(rec_id)

    # Print out field/column @ids for each record set
    try:
        record_set_obj = dataset.metadata._lookup_by_id(rec_id)
        fields = getattr(record_set_obj, 'field', None)
        if fields is not None:
            print('    Fields (by @id):')
            if isinstance(fields, list):
                for f in fields:
                    field_id = f.get('@id', None) if isinstance(f, dict) else str(f)
                    print(f'      - {field_id}')
            else:
                print(f'      - {fields.get("@id", None)}')
        columns = getattr(record_set_obj, 'column', None)
        if columns is not None:
            print('    Columns (by @id):')
            if isinstance(columns, list):
                for c in columns:
                    col_id = c.get('@id', None) if isinstance(c, dict) else str(c)
                    print(f'      - {col_id}')
            else:
                print(f'      - {columns.get("@id", None)}')
    except Exception as e:
        print(f'    Could not retrieve columns for record set {rec_id}: {e}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. If you don't know the column names beforehand, you can print the first record as a dict to inspect the keys.

In [ ]:
# Let's extract records from the main record set.
# For this dataset, the Croissant schema typically provides a single major data record set.
if len(all_record_set_ids) == 0:
    raise ValueError('No record sets detected. Adjust extraction logic as appropriate for the dataset situation.')

# Use the first discovered record set as the main record set
main_record_set_id = all_record_set_ids[0]
print(f"Loading records from record set @id: {main_record_set_id}")

df = pd.DataFrame(list(dataset.records(record_set=main_record_set_id)))
# Show the discovered columns
print('Discovered columns:')
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Inspect numeric fields in the main DataFrame
print("Numeric fields in this record set (inferred):")
numeric_fields = []
for c in df.columns:
    # Quick check for numeric data
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_fields.append(c)
        print(f"  - {c}")
# Heuristically pick the first numeric field for demonstration
if not numeric_fields:
    print('No numeric fields detected. Please adjust the code after inspecting df.head() above.')
else:
    numeric_field_id = numeric_fields[0]  # Use as @id
    print(f'Using {numeric_field_id} for filtering and normalization.')
    threshold = df[numeric_field_id].mean()  # Example threshold: greater than mean
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (mean):")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a non-numeric (likely categorical) field
    group_field_candidates = [c for c in df.columns if pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_object_dtype(df[c])]
    if group_field_candidates:
        group_field = group_field_candidates[0]  # Pick first possible grouping field
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot if there is another numeric field
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6,6))
        sns.scatterplot(data=df, x=numeric_fields[0], y=numeric_fields[1])
        plt.title(f'Scatter: {numeric_fields[0]} vs {numeric_fields[1]}')
        plt.show()
else:
    print('No numeric_field_id selected for plotting.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the Mass Spectrometry dataset using its Croissant schema and explored its key record sets and fields (referenced by `@id`).
- Using the `mlcroissant` library, we loaded main records into a DataFrame, listed and analyzed their columns and selected numeric fields for demonstration.
- Simple filtering, normalization, grouping, and visualization steps provide a foundation for further biological or statistical analysis on this mass spectrometry dataset.

Continue to iterate on these analyses to derive scientific or data-driven insights!